# EvalForge — reproducible GPU experiment
Enable internet and a CUDA GPU. This notebook creates a Python 3.11 environment independently of Kaggle’s host Python. The data mirror is pinned and checksum verified; model and dataset terms apply. No secret tokens are required for the public model and data. Outputs contain the adapter, manifest, and paired evaluation report.
Before running: select **GPU T4 x2**, enable Internet, and set Persistence to **Files** (or **Variables & Files** if offered). Use **Save Version → Save & Run All** for the final run so completed output files are saved with the version. After it succeeds, download and verify the ZIP before deleting or resetting anything.


In [1]:
import subprocess, sys, os
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'venv', '--python', '3.11', '--seed', '/kaggle/temp/evalforge-env'], check=True)
repo = Path('/kaggle/working/EvalForge')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Navaneeth-007/EvalForge.git', str(repo)], check=True)
print(subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True))
os.environ['HF_HOME'] = '/kaggle/temp/evalforge-model-cache'
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
os.environ['WANDB_MODE'] = 'disabled'
subprocess.run(['uv', 'pip', 'install', '--python', '/kaggle/temp/evalforge-env/bin/python', '-r', str(repo / 'requirements-ml.txt')], check=True)


Using CPython 3.11.17
Creating virtual environment with seed packages at: /kaggle/temp/evalforge-env
 + packaging==26.3
 + pip==26.2.2
 + setuptools==84.2.1
 + wheel==0.48.0
Activate with: source /kaggle/temp/evalforge-env/bin/activate
Cloning into '/kaggle/working/EvalForge'...


613a4b9291ab8e67b88d07f5e15b35486cb53091



Using Python 3.11.17 environment at: /kaggle/temp/evalforge-env
Resolved 116 packages in 911ms
Prepared 115 packages in 37.96s
Uninstalled 1 package in 1ms
Installed 115 packages in 337ms
 + accelerate==1.1.1
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + altair==5.5.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.15.1
 + attrs==26.1.0
 + bert-score==0.3.13
 + bitsandbytes==0.44.1
 + blinker==1.9.0
 + cachetools==5.5.2
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + click==8.5.0
 + colorama==0.4.6
 + contourpy==1.3.3
 + cycler==0.12.1
 + datasets==3.1.0
 + dill==0.3.8
 + docker-pycreds==0.4.0
 + fastapi==0.162.0
 + filelock==4.2.4
 + fonttools==4.68.0
 + frozenlist==1.8.0
 + fsspec==2024.9.0
 + gitdb==4.0.12
 + gitpython==3.2.1
 + h11==0.16.0
 + httptools==0.9.0
 + huggingface-hub==0.26.5
 + idna==3.20
 + jinja2==3.1.6
 + jsonschema==4.26.0
 + jsonschema-specifications==2025.9.1
 + kiwisolver==1.5.1
 + lxml==6.1.3
 + markdown-it-py==4.2.0
 +

CompletedProcess(args=['uv', 'pip', 'install', '--python', '/kaggle/temp/evalforge-env/bin/python', '-r', '/kaggle/working/EvalForge/requirements-ml.txt'], returncode=0)

## Train and evaluate
Full training split, two epochs, two GPUs, and 200 development questions. Distributed training uses batch size 1 per GPU and gradient accumulation 8: effective batch size remains 16. Data preparation and paired evaluation each run once. Start with a short smoke run in a separate notebook if desired; no two-GPU hardware verification has been performed by the maintainer.


In [2]:
RUN_ARGS = ['--gpus', '2', '--epochs', '2', '--eval-n', '200']
subprocess.run(['/kaggle/temp/evalforge-env/bin/python', '-u', '-m', 'scripts.run_gpu', *RUN_ARGS], cwd=str(repo), env=os.environ.copy(), check=True)
import shutil
shutil.copy2(repo / 'data/processed/eval_samples.json', repo / 'outputs/eval_samples.json')
archive = shutil.make_archive('/kaggle/working/evalforge-evidence', 'zip', repo / 'outputs')
print('DOWNLOAD AND VERIFY:', archive)


INFO:datasets:PyTorch version 2.5.1 available.
INFO:train.dataset_prep:Loading Spider dataset from Hugging Face...
Generating validation split: 100%|██████████| 1034/1034 [00:00<00:00, 159281.27 examples/s]
INFO:train.dataset_prep:Train: 7000 examples | Validation: 1034 examples
INFO:train.dataset_prep:Formatting prompts...
Formatting: 100%|██████████| 1034/1034 [00:00<00:00, 13629.13 examples/s]
INFO:train.dataset_prep:Loading tokenizer: microsoft/Phi-3-mini-4k-instruct
Saving the dataset (1/1 shards): 100%|██████████| 1034/1034 [00:00<00:00, 73336.66 examples/s]
INFO:train.dataset_prep:Dataset saved to ./data/processed
INFO:train.dataset_prep:Saved 200 eval samples to data/processed/eval_samples.json
*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
************************************

trainable params: 25,165,824 || all params: 3,846,245,376 || trainable%: 0.6543
trainable params: 25,165,824 || all params: 3,846,245,376 || trainable%: 0.6543


INFO:train.train:Starting training...
INFO:train.train:Starting training...
  1%|          | 10/864 [03:14<3:54:53, 16.50s/it]

{'loss': 0.8342, 'grad_norm': 0.9826923608779907, 'learning_rate': 7.692307692307693e-05, 'epoch': 0.02}


  2%|▏         | 20/864 [05:13<2:42:30, 11.55s/it]

...

{'train_runtime': 16191.1994, 'train_samples_per_second': 0.855, 'train_steps_per_second': 0.053, 'train_loss': 0.11003561273079228, 'epoch': 2.0}


INFO:train.train:Training complete.
INFO:eval.run_eval:Loading eval samples from data/processed/eval_samples.json
INFO:eval.run_eval:Evaluating on 200 examples
INFO:eval.run_eval:
INFO:eval.run_eval:EVALUATING BASE MODEL (before fine-tuning)
INFO:eval.run_eval:============================================================
INFO:eval.run_eval:Loading model: microsoft/Phi-3-mini-4k-instruct
Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.64s/it]
INFO:eval.run_eval:  Generated 4/200
INFO:eval.run_eval:  Generated 8/200
INFO:eval.run_eval:  Generated 12/200
INFO:eval.run_eval:  Generated 16/200
INFO:eval.run_eval:  Generated 20/200
INFO:eval.run_eval:  Generated 24/200
INFO:eval.run_eval:  Generated 28/200
INFO:eval.run_eval:  Generated 32/200
INFO:eval.run_eval:  Generated 36/200
INFO:eval.run_eval:  Generated 40/200
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
INFO:eval.run_eval:  Generated 44/200
INFO:eval.run


EVALFORGE — BEFORE vs AFTER RESULTS
Metric                         Base Model    Fine-tuned           Δ
─────────────────────────────────────────────────────────────────
Exact Match Accuracy               0.1450        0.2300   (+0.0850)
Execution Accuracy                 0.5000        0.5900   (+0.0900)
BLEU-4                             0.3076        0.6979   (+0.3903)
BERTScore F1                          N/A           N/A            
─────────────────────────────────────────────────────────────────

📋 Copy the above table into your README.md

Experiment complete: preserve outputs/ and update the model card with measured evidence.


INFO:eval.run_eval:
Full results saved to outputs/eval_results.json


DOWNLOAD AND VERIFY: /kaggle/working/evalforge-evidence.zip


## Inspect and save evidence
These are development metrics, not a blind test score. Inspect incorrect predictions before claiming an improvement. Do not publish the model card until its pending fields have been replaced with measured values.


In [1]:
import json, shutil
manifest = json.loads((repo / 'outputs/phi3-sql-qlora/training_manifest.json').read_text())
report = json.loads((repo / 'outputs/eval_results.json').read_text())
print('Training examples:', manifest['train_examples'])
print('GPU:', manifest['gpu'], 'Elapsed seconds:', manifest['elapsed_seconds'])
print('Base metrics:', report['base_model_metrics'])
print('Adapter metrics:', report['finetuned_model_metrics'])
import hashlib, zipfile
archive_path = Path('/kaggle/working/evalforge-evidence.zip')
with zipfile.ZipFile(archive_path) as z:
    assert z.testzip() is None, 'Corrupt archive'
    assert any(name.endswith('adapter_model.safetensors') for name in z.namelist())
print('Archive bytes:', archive_path.stat().st_size)
print('SHA256:', hashlib.file_digest(archive_path.open('rb'), 'sha256').hexdigest())


Training examples: 6918
GPU: Tesla T4 Elapsed seconds: 16274.902881199
Base metrics: {'exact_match': 0.145, 'exact_match_details': {'exact_match_accuracy': 0.145, 'num_correct': 29, 'num_total': 200, 'per_example': [{'prediction': 'SELECT COUNT(*) FROM singer;', 'gold': 'SELECT count(*) FROM singer', 'prediction_normalised': 'select count(*) from singer', 'gold_normalised': 'select count(*) from singer', 'match': True}, {'prediction': 'SELECT COUNT(*) FROM singer;', 'gold': 'SELECT count(*) FROM singer', 'prediction_normalised': 'select count(*) from singer', 'gold_normalised': 'select count(*) from singer', 'match': True}, {'prediction': 'SELECT Name, Country, Age\nFROM singer\nORDER BY Age DESC;', 'gold': 'SELECT name ,  country ,  age FROM singer ORDER BY age DESC', 'prediction_normalised': 'select name, country, age from singer order by age desc', 'gold_normalised': 'select name , country , age from singer order by age desc', 'match': False}, {'prediction': 'SELECT Name, Country, A